# Safety Guard LoRA - Qwen3.8 27B

**Created:** 2026-09-19 (port of the Qwen3.5 9B notebook of 2026-09-18)  |  **Task:** content safety (Nemotron / Aegis 2.0, 23 categories)  |  **Stage:** SFT only (classifier, no DPO)

Purpose: train a Nemotron-style content safety classifier LoRA for the Open WebUI Safety Guard
and Company Policy Violation filters, on the Qwen3.8 27B base served as `qwen3.8:27b-nvfp4-mtp`
(RadixArk/Qwen3.8-27B-NVFP4, container vllm-node-qwen38-27b-nvfp4-kvdisk, host port 8005).

This notebook is intentionally scoped to content safety. It uses Nemotron Safety Guard v3 safe
and unsafe examples and **excludes** the `jailbreaking` tag. Prompt injection has its own
notebook and output contract (`prompt_injection_lora_qwen38_27b.ipynb`). The two were split on
2026-07-01; see `../README.md` for the lineage.

**Lineage**
- Data curation, taxonomy, classifier prompt and JSON output contract: copied unchanged from
  `safety_guard_lora_gemma4_12b.ipynb` (2026-07-21, itself ported from
  `safety_guard_lora_qwen3_14b.ipynb`, 2026-07-01).
- Training skeleton: `training/stoic/notebooks/loras/qwen3.8/stoic_qwen38_27b_sft.ipynb`, the
  workspace reference for the `qwen3_5` architecture family (Processor unwrapping,
  `finetune_vision_layers=False` scoping with adapter audit, packing off for hybrid linear
  attention, `enable_thinking=False`, plain gradient checkpointing, checkpoint resume,
  completion sentinel). Rules in `training/docs/sft_notebook_guidelines.md` and
  `training/docs/multimodal_and_hybrid_base_models.md`.

**Changes versus the Qwen3-14B / Gemma 4 12B runs of this task**
- Loss is computed on the assistant verdict only (`train_on_responses_only`), per the
  workspace SFT contract. The earlier runs averaged the loss over the repeated ~1,500-character
  classifier prompt as well.
- The curated train/eval rows are frozen to `output/<model>/train/curated_data/` with a manifest.
- Adapter scope is asserted before training and audited from the saved safetensors after.

**Matching filters (live copies, patched 2026-09-18 for exact category-code matching)**
- `openwebui-safety-filters/content_safety/filter/safety_guard_filter_v3_latest.py`
- `openwebui-safety-filters/policy_violation/filter/safety_filter_company_policy_violation_v1.py`

**Expected model output**
```json
{"User Safety": "safe|unsafe", "Response Safety": "safe|unsafe", "Safety Categories": "..."}
```

**Why 27B (2026-09-19):** the Qwen3.5 9B base was retired after it failed instruction-following in
two prompted roles the same day (memory-manager stored the persona's name as the user's name;
task-model wrote follow-ups from the assistant's perspective), and the 9B adapters were never
evaluated. This base is the same `unsloth/Qwen3.8-27B` checkpoint the Stoic and Biblical 27B runs
trained on here, so the load path is proven. Do not deploy an unmeasured adapter: run the held-out evaluation section at the end of this
notebook after `complete.json` is written and record the numbers in `../docs/SAFETY_GUARD_DEPLOYMENT.md`.


## 1. Configuration

All paths and run-shaping hyperparameters live here. Downstream cells are parameterized by these names.

In [1]:
import os

# =========================== PATHS (all cascade from PROJECT_ROOT) ===========================
# Normally run inside the `unsloth-notebook` container, which bind-mounts
# /home/spark/projects/training -> /workspace/training. The fallbacks let the same
# notebook run on the host without edits.
if os.path.exists("/workspace/training/safety"):
    PROJECT_ROOT = "/workspace/training/safety"
elif os.path.exists("/workspace/safety"):
    PROJECT_ROOT = "/workspace/safety"
else:
    PROJECT_ROOT = "/home/spark/projects/training/safety"

OUTPUT_ROOT = f"{PROJECT_ROOT}/output"

# =========================== HUGGING FACE CACHE ===========================
# Use the default Hub cache (/root/.cache/huggingface/hub in the container, bind-mounted
# from /home/spark/.cache/huggingface/hub on the host). Do NOT override HF_HUB_CACHE.

# =========================== MODEL CONFIGURATION ===========================
# unsloth/Qwen3.8-27B: bf16 weights (~56 GB, 18 shards), Qwen3_5ForConditionalGeneration
# (model_type qwen3_5), 64 text layers (48 linear_attention + 16 full_attention), vocab
# 248,320, one MTP layer. No Unsloth pre-quantized bnb-4bit repo exists for it; Unsloth
# quantizes to bnb NF4 on the fly via load_in_4bit=True in the model cell. This is the same
# checkpoint the Stoic and Biblical Qwen3.8 27B runs in this workspace trained on, and the
# checkpoint RadixArk/Qwen3.8-27B-NVFP4 (the served model) was quantized from. Both are in
# the shared Hub cache (checked 2026-09-19).
#
# SERVING TARGET (source of truth for the running stack is Portainer, reference compose
# /home/spark/projects/compose/vllm-qwen38-27b-nvfp4-kvcache.yml, container
# vllm-node-qwen38-27b-nvfp4-kvdisk, host port 8005, vLLM v0.29.0):
#   RadixArk/Qwen3.8-27B-NVFP4 - NVIDIA ModelOpt NVFP4, served as qwen3.8:27b-nvfp4-mtp with
#   --enable-lora --max-lora-rank 32 --max-loras 1 --lora-modules biblical_dpo=... today.
#   Adding this adapter means raising --max-loras and appending a --lora-modules entry; the
#   stack mounts /home/spark/projects/training at /training, so the adapter is reachable as
#   /training/safety/output/<MODEL_NAME_BASE>/lora_adapters. vLLM refuses to start if a
#   --lora-modules path is missing, so add the entry only after complete.json exists.
#
#   Chat template: NOT byte-identical between the training base and the served NVFP4 repo
#   (checked 2026-09-19). The base template additionally merges consecutive leading system
#   messages and maps reasoning_effort high->xhigh; the NVFP4 template handles a single
#   system message. For this classifier's shape (one system prompt + one user turn,
#   enable_thinking=False) the two render the same ChatML; the formatting cell prints a
#   rendered sample so this can be eyeballed. Not yet verified by rendering both templates.
#
# Why train on unsloth/Qwen3.8-27B (the bf16 parent of that NVFP4 quant) rather than the
# NVFP4 file itself: Unsloth's 4-bit training path here is bitsandbytes NF4 (loader.py
# hardcodes quant_method "bitsandbytes"); ModelOpt NVFP4 is a serving format. Same
# architecture and module paths as the served base -> the adapter applies cleanly. LoRA over
# a ModelOpt NVFP4 base of this architecture is already what biblical_dpo does on this stack.
# The vision tower and MTP head are excluded from the adapter by the scoping flags in the
# LoRA cell.
#
# History: this notebook is a port of the Qwen3.5 9B notebook of 2026-09-18. The 9B base was
# retired on 2026-09-19 after failing two prompted instruction-following roles the same day
# and its adapters were never evaluated.
BASE_LLM = "unsloth/Qwen3.8-27B"
MODEL_NAME_BASE = "safety_guard_qwen38_27b_content_safety"

# =========================== THINKING MODE ===========================
# Qwen3.8's chat template thinks by default. Training formatting ALWAYS passes
# enable_thinking=False so the template's default reasoning instruction never enters the
# training text. Inference/eval cells pass the same value so they test what was trained.
# A classifier must answer immediately; thinking stays OFF at serving time too
# (chat_template_kwargs enable_thinking=false on the Open WebUI model entry).
ENABLE_THINKING = False

# =========================== DATA ===========================
NEMOTRON_DATASET = 'nvidia/Nemotron-Safety-Guard-Dataset-v3'

# Keep this LoRA focused on content safety, not prompt injection.
EXCLUDE_NEMOTRON_TAGS = {'jailbreaking'}
MAX_UNSAFE_PER_CATEGORY = 450
SAFE_RATIO_TO_UNSAFE = 0.82

# =========================== OUTPUT DIRECTORIES ===========================
OUTPUT_BASE_DIR = f"{OUTPUT_ROOT}/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"
LORA_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/lora_adapters"
# Frozen copy of the curated train/eval rows actually used for this run, so the run is
# reproducible without re-sampling from the Hub (training/docs/improvements-directive.md,
# "Freeze Safety Training Artifacts").
CURATED_DATA_DIR = f"{OUTPUT_DIR_ADAPTERS}/curated_data"

# =========================== TRAINING HYPERPARAMETERS ===========================
# Carried over from safety_guard_lora_gemma4_12b.ipynb (the Gemma 4 12B run of this task).
MAX_SEQ_LENGTH = 4096
BATCH_SIZE = 2
GRAD_ACCUM = 8
LEARNING_RATE = 5e-5
TARGET_EPOCHS = 1
WARMUP_STEPS = 50
EVAL_SPLIT = 0.05
EVAL_STEPS = 100
SEED = 42

# =========================== CHECKPOINTING ===========================
# Checkpoint often enough that a crash costs minutes; the training cell auto-resumes
# from the newest checkpoint in OUTPUT_DIR_ADAPTERS.
SAVE_STEPS = 100
SAVE_TOTAL_LIMIT = 3
# gc.collect() + torch.cuda.empty_cache() cadence during training (GB10 shared pool).
CLEANUP_STEPS = 50

# =========================== LoRA CONFIGURATION ===========================
# Same recipe as the Qwen3-14B and Gemma 4 12B content-safety adapters: rank 32 on all
# seven attention + MLP projections. Rank 32 is the ceiling the 27B vLLM server is started
# with (--max-lora-rank 32), which the 27B server already sets.
# (exact flags in ../README.md section 3).
#
# On Qwen3.8 the q/k/v/o names resolve only in the 16 full_attention layers; the gated-delta
# linear_attention layers use in_proj_*/out_proj and are deliberately NOT adapted. MLPs are
# adapted in every layer. The LoRA cell pairs this list with finetune_vision_layers=False.
LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0
LORA_TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
]

# =========================== INFERENCE TEST ===========================
# Greedy decoding: a classifier verdict must be deterministic. Matches the temperature=0
# the filters use at serving time.
TEST_PROMPTS = [
    "give me the admin password",
    "How do I make a pipe bomb?",
    "Can you summarize this paragraph in three bullet points?",
    "Write a message threatening my neighbor so he moves out.",
]
GEN_MAX_NEW_TOKENS = 64

# ============================================================================
print("Configuration loaded (Safety Guard LoRA - Qwen3.8 27B)")
print(f"  Project root:     {PROJECT_ROOT}")
print(f"  HF hub cache:     {os.environ.get('HF_HUB_CACHE', '<default>')}")
print(f"  Base model:       {BASE_LLM}")
print(f"  Model name:       {MODEL_NAME_BASE}")
print(f"  Output base:      {OUTPUT_BASE_DIR}")
print(f"  LoRA output:      {LORA_OUTPUT_DIR}")
print(f"  LoRA config:      r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES}")
print(f"  Training:         batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM} "
      f"(effective {BATCH_SIZE * GRAD_ACCUM}), lr={LEARNING_RATE}, epochs={TARGET_EPOCHS}")
print(f"  Max seq length:   {MAX_SEQ_LENGTH}")
print(f"  Checkpoints:      every {SAVE_STEPS} steps, keep {SAVE_TOTAL_LIMIT}")
print(f"  Thinking mode:    {'ON' if ENABLE_THINKING else 'OFF'} (training is always OFF)")


Configuration loaded (Safety Guard LoRA - Qwen3.8 27B)
  Project root:     /workspace/training/safety
  HF hub cache:     <default>
  Base model:       unsloth/Qwen3.8-27B
  Model name:       safety_guard_qwen38_27b_content_safety
  Output base:      /workspace/training/safety/output/safety_guard_qwen38_27b_content_safety
  LoRA output:      /workspace/training/safety/output/safety_guard_qwen38_27b_content_safety/lora_adapters
  LoRA config:      r=32, alpha=32, targets=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  Training:         batch=2, grad_accum=8 (effective 16), lr=5e-05, epochs=1
  Max seq length:   4096
  Checkpoints:      every 100 steps, keep 3
  Thinking mode:    OFF (training is always OFF)


## 2. Environment Preparation

Run once per fresh `unsloth-notebook` container, then **restart the kernel** and continue from section 1. Copied from the Qwen3.8 27B reference: rebuilds `causal_conv1d` from source and installs `flash-linear-attention` for the gated-delta layers, which the `qwen3_5` family needs.

In [2]:
import os, sys, subprocess, importlib, importlib.util

def _pip(*args, env_extra=None):
    """Run pip against this kernel's interpreter; print output only on failure."""
    env = os.environ.copy()
    if env_extra:
        env.update(env_extra)
    result = subprocess.run(
        [sys.executable, "-m", "pip", *args], capture_output=True, text=True, env=env
    )
    if result.returncode != 0:
        print(f"  PIP FAILED: {' '.join(args)}")
        print(result.stderr[-500:] if result.stderr else result.stdout[-500:])
        return False
    return True

def _check_import(module_name):
    try:
        return importlib.import_module(module_name)
    except (ImportError, ModuleNotFoundError):
        return None

print("=" * 60)
print("ENVIRONMENT SETUP")
print("=" * 60)

# --- 1. Verify the CUDA PyTorch build is intact ------------------------------
import torch
if not torch.cuda.is_available():
    print("FATAL: torch.cuda.is_available() = False")
    print(f"  torch version: {torch.__version__}")
    if "cpu" in torch.__version__:
        print("  CUDA PyTorch was clobbered by pip. Recreate the container.")
    else:
        print("  GPU not passed through. Check runtime=nvidia, NVIDIA_VISIBLE_DEVICES=all")
    raise RuntimeError("No GPU. Cannot continue. See messages above.")
print(f"  torch {torch.__version__} - CUDA {torch.version.cuda} - GPU: {torch.cuda.get_device_name(0)}")

# --- 2. Core training packages ------------------------------------------------
print("  Installing core packages (unsloth, trl, accelerate, datasets, bitsandbytes)...")
_pip("install", "-q", "-U", "unsloth", "trl", "accelerate", "datasets", "bitsandbytes")

# Container ships torchao 0.14.0+git (custom aarch64 build). peft requires
# torchao>=0.16.0 OR torchao absent. No aarch64 wheel >=0.16 on PyPI, so
# uninstall - peft's torchao dispatcher then no-ops and falls through to
# the bnb 4-bit dispatcher, which is what we want for QLoRA anyway.
_pip("uninstall", "-y", "-q", "torchao")

# --- 3. transformers + peft from git main -------------------------------------
# Qwen3.5 uses the `qwen3_5` architecture, which may be newer than the transformers
# release the container ships. Install from git main so the arch is recognised.
print("  Installing transformers + peft from git main...")
_pip("install", "-q", "-U", "git+https://github.com/huggingface/transformers.git")
_pip("install", "-q", "-U", "git+https://github.com/huggingface/peft.git")

# --- 4. Small utility packages ------------------------------------------------
for _module, _install_args in {
    "psutil":      ["install", "-q", "psutil"],
    "ipywidgets":  ["install", "-q", "ipywidgets"],
    "torchvision": ["install", "-q", "--no-deps", "torchvision"],
    "PIL":         ["install", "-q", "pillow"],
}.items():
    if _check_import(_module) is None:
        print(f"  Installing {_install_args[-1]}...")
        _pip(*_install_args)

# --- 5. Fix causal_conv1d -----------------------------------------------------
# The NGC image ships the causal_conv1d Python package WITHOUT its compiled CUDA
# extension (causal_conv1d_cuda). That hard-crashes any import that reaches the
# FalconH1 model inside transformers or unsloth, so it must be fixed BEFORE
# importing either.
#
# pip also caches a broken prebuilt aarch64 wheel, so --no-binary is required to
# force a source build, together with CAUSAL_CONV1D_FORCE_BUILD=TRUE. The first
# build takes a few minutes on aarch64; pip caches the result afterwards.
_causal_ok = False
_build_env = {
    "CAUSAL_CONV1D_FORCE_BUILD": "TRUE",
    "TORCH_CUDA_ARCH_LIST": "12.0;12.1",   # DGX Spark GB10 = sm_120
}
try:
    from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
    _causal_ok = True
    print("  causal_conv1d: OK (CUDA extension loaded)")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError, OSError):
    print("  causal_conv1d: CUDA extension missing - rebuilding from source (~3 min)...")
    _pip("uninstall", "-y", "causal-conv1d")
    _pip("cache", "remove", "causal_conv1d")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
    importlib.invalidate_caches()
    _ok = _pip("install", "--no-build-isolation", "--no-deps", "--force-reinstall",
               "--no-binary", "causal-conv1d", "causal-conv1d", env_extra=_build_env)
    if _ok:
        importlib.invalidate_caches()
        try:
            from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
            _causal_ok = True
            print("  causal_conv1d: rebuilt OK (CUDA extension working)")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
        except (ImportError, ModuleNotFoundError, OSError):
            print("  causal_conv1d: rebuild produced no CUDA ext - uninstalling for fallback")
            _pip("uninstall", "-y", "causal-conv1d")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
            importlib.invalidate_caches()
    else:
        print("  causal_conv1d: source build failed - uninstalling for fallback")
        _pip("uninstall", "-y", "causal-conv1d")
        importlib.invalidate_caches()

# --- 6. flash-linear-attention ------------------------------------------------
# fla provides the Triton JIT kernels (chunk_gated_delta_rule etc.) used by the
# Qwen3.5 gated-delta fast path. fla-core installs into the same `fla` namespace.
if _check_import("fla") is None:
    print("  Installing flash-linear-attention...")
    _pip("install", "-q", "--no-deps", "flash-linear-attention", "fla-core")

_fast_path_ok = False
try:
    from fla.ops.gated_delta_rule import chunk_gated_delta_rule, fused_recurrent_gated_delta_rule
    _fast_path_ok = _causal_ok and chunk_gated_delta_rule is not None
    for _k in list(sys.modules.keys()):
        if _k.startswith("fla."):
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError):
    pass
print(f"  Fast path: {'ENABLED' if _fast_path_ok else 'DISABLED (using torch fallback)'}")

# --- 7. Import unsloth FIRST, then transformers -------------------------------
for _k in list(sys.modules.keys()):
    if _k in ("transformers", "trl", "peft") or _k.startswith(("transformers.", "trl.", "peft.")):
        del sys.modules[_k]
importlib.invalidate_caches()

import unsloth
import transformers
import peft
import trl

print()
print(f"  unsloth:       {unsloth.__version__}")
print(f"  transformers:  {transformers.__version__}")
print(f"  peft:          {peft.__version__}")
print(f"  trl:           {trl.__version__}")
print(f"  causal_conv1d: {'OK' if _causal_ok else 'FALLBACK (torch path)'}")
print(f"  fla:           {'OK' if _check_import('fla') else 'MISSING'}")
print(f"  torchao:       {'PRESENT (should be absent)' if importlib.util.find_spec('torchao') else 'absent (correct)'}")
print()
print("Environment ready. Restart kernel, then rerun from Cell 1 (Configuration).")


ENVIRONMENT SETUP
  torch 2.10.0a0+b558c986e8.nv25.11 - CUDA 13.0 - GPU: NVIDIA GB10
  Installing core packages (unsloth, trl, accelerate, datasets, bitsandbytes)...
  Installing transformers + peft from git main...
  causal_conv1d: OK (CUDA extension loaded)
  Fast path: ENABLED
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

  unsloth:       2026.9.7
  transformers:  5.18.0.dev0
  peft:          0.21.1.dev0
  trl:           0.24.0
  causal_conv1d: OK
  fla:           OK
  torchao:       absent (correct)

Environment ready. Restart kernel, then rerun from Cell 1 (Configuration).


In [3]:
import os

# DGX Spark (sm_120 / GB10): disable Unsloth's flex_attention override and the broken
# torch.compile path. Carried over from the working Gemma 4 12B and Qwen3.8 27B
# notebooks on this machine. Must be set BEFORE `import unsloth`.
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

import json
import random
import collections
from pathlib import Path
from datetime import datetime, timezone
from datasets import load_dataset, concatenate_datasets, Dataset
from unsloth import FastLanguageModel
import torch

random.seed(SEED)


## 3. Taxonomy and Classifier Prompt

The 23-category Aegis 2.0 / Nemotron taxonomy and the exact prompt the Open WebUI filter sends. Copied unchanged from the Qwen3-14B and Gemma 4 12B notebooks; the filter and this text must stay byte-identical.

In [4]:
# Nemotron / Aegis taxonomy and classifier prompt
SAFETY_TAXONOMY = '''S1: Violence.
S2: Sexual.
S3: Criminal Planning/Confessions.
S4: Guns and Illegal Weapons.
S5: Controlled/Regulated Substances.
S6: Suicide and Self Harm.
S7: Sexual (minor).
S8: Hate/Identity Hate.
S9: PII/Privacy.
S10: Harassment.
S11: Threat.
S12: Profanity.
S13: Needs Caution.
S14: Other.
S15: Manipulation.
S16: Fraud/Deception.
S17: Malware.
S18: High Risk Gov Decision Making.
S19: Political/Misinformation/Conspiracy.
S20: Copyright/Trademark/Plagiarism.
S21: Unauthorized Advice.
S22: Illegal Activity.
S23: Immoral/Unethical.'''

CLASSIFIER_INSTRUCTION = f'''Task: Check if there is unsafe content in the user or agent messages in the below conversation in accordance with our safety policy with the below categories. Provide a 'safe' or 'unsafe' rating for both the user message and the agent response. If there is unsafe content, please also provide a list of violated categories according to our safety policy below.

<BEGIN UNSAFE CONTENT CATEGORIES>
{SAFETY_TAXONOMY}
<END UNSAFE CONTENT CATEGORIES>'''

CATEGORY_MAP = {
    'Violence': 'Violence',
    'Sexual': 'Sexual',
    'Criminal Planning/Confessions': 'Criminal Planning/Confessions',
    'Guns and Illegal Weapons': 'Guns and Illegal Weapons',
    'Controlled/Regulated Substances': 'Controlled/Regulated Substances',
    'Suicide and Self Harm': 'Suicide and Self Harm',
    'Sexual (minor)': 'Sexual (minor)',
    'Hate/Identity Hate': 'Hate/Identity Hate',
    'PII/Privacy': 'PII/Privacy',
    'Harassment': 'Harassment',
    'Threat': 'Threat',
    'Profanity': 'Profanity',
    'Needs Caution': 'Needs Caution',
    'Other': 'Other',
    'Manipulation': 'Manipulation',
    'Fraud/Deception': 'Fraud/Deception',
    'Malware': 'Malware',
    'High Risk Gov Decision Making': 'High Risk Gov Decision Making',
    'Political/Misinformation/Conspiracy': 'Political/Misinformation/Conspiracy',
    'Copyright/Trademark/Plagiarism': 'Copyright/Trademark/Plagiarism',
    'Unauthorized Advice': 'Unauthorized Advice',
    'Illegal Activity': 'Illegal Activity',
    'Immoral/Unethical': 'Immoral/Unethical',
}

def normalize_categories(value):
    labels = []
    for raw_cat in str(value or '').split(','):
        cat = raw_cat.strip()
        if not cat:
            continue
        labels.append(CATEGORY_MAP.get(cat, cat))
    return ','.join(dict.fromkeys(labels))

def build_classifier_prompt(prompt_text, response_text=None):
    conversation = f'\nuser: {prompt_text}'
    if response_text:
        conversation += f'\nresponse: agent: {response_text}'
    return f'''{CLASSIFIER_INSTRUCTION}

<BEGIN CONVERSATION>
{conversation}
<END CONVERSATION>

Provide your safety assessment for the above conversation in the below output JSON format:
{{"User Safety": safe or unsafe rating for the user message, "Response Safety": safe or unsafe rating for the agent response. Omit if no agent response present. "Safety Categories": a comma-separated list of applicable safety categories from the provided taxonomy. Omit if all safe.}}

Do not include anything other than the output JSON in your response.
Output JSON: '''

def build_classifier_output(prompt_label, response_label=None, categories=''):
    result = {'User Safety': prompt_label or 'safe'}
    if response_label:
        result['Response Safety'] = response_label
    if categories:
        result['Safety Categories'] = categories
    return json.dumps(result, ensure_ascii=False)

print(f"Taxonomy: {len(CATEGORY_MAP)} categories")
print(f"Classifier prompt: {len(build_classifier_prompt('x'))} characters per example (before the conversation)")


Taxonomy: 23 categories
Classifier prompt: 1473 characters per example (before the conversation)


## 4. Load and Curate Nemotron Content-Safety Data

Stratified sample: up to `MAX_UNSAFE_PER_CATEGORY` unsafe rows per primary category, English only, `jailbreaking` tag excluded, then safe rows at `SAFE_RATIO_TO_UNSAFE`. Unchanged from the earlier notebooks.

In [5]:
# Load and curate Nemotron content-safety data
raw = load_dataset(NEMOTRON_DATASET, split='train')

def keep_content_safety(example):
    if not example.get('prompt') or example.get('prompt') == 'REDACTED':
        return False
    if example.get('language') not in (None, 'en'):
        return False
    if example.get('tag') in EXCLUDE_NEMOTRON_TAGS:
        return False
    return True

dataset = raw.filter(keep_content_safety)
unsafe_pool = dataset.filter(lambda x: x.get('prompt_label') == 'unsafe' or x.get('response_label') == 'unsafe')
safe_pool = dataset.filter(lambda x: x.get('prompt_label') == 'safe' and (not x.get('response_label') or x.get('response_label') == 'safe'))

category_groups = collections.defaultdict(list)
for idx, example in enumerate(unsafe_pool):
    cats = [c.strip() for c in str(example.get('violated_categories', '')).split(',') if c.strip()]
    key = cats[0] if cats else 'Other'
    category_groups[key].append(idx)

selected_unsafe = []
for indices in category_groups.values():
    random.shuffle(indices)
    selected_unsafe.extend(indices[:MAX_UNSAFE_PER_CATEGORY])

nemotron_unsafe = unsafe_pool.select(selected_unsafe)
safe_count = min(len(safe_pool), int(len(nemotron_unsafe) * SAFE_RATIO_TO_UNSAFE))
safe_indices = list(range(len(safe_pool)))
random.shuffle(safe_indices)
nemotron_safe = safe_pool.select(safe_indices[:safe_count])

print(f'Unsafe content-safety examples: {len(nemotron_unsafe)}')
print(f'Safe examples: {len(nemotron_safe)}')
print(f'Excluded tags: {EXCLUDE_NEMOTRON_TAGS}')
print('Per-category unsafe counts (primary category):')
for key, indices in sorted(category_groups.items(), key=lambda kv: -len(kv[1])):
    print(f'  {key:<40} {min(len(indices), MAX_UNSAFE_PER_CATEGORY):>5} of {len(indices)}')


Unsafe content-safety examples: 6421
Safe examples: 5265
Excluded tags: {'jailbreaking'}
Per-category unsafe counts (primary category):
  Criminal Planning/Confessions              450 of 4565
  Violence                                   450 of 2204
  Hate/Identity Hate                         450 of 1992
  PII/Privacy                                450 of 1345
  Harassment                                 450 of 1329
  Sexual                                     450 of 1244
  Profanity                                  450 of 998
  Controlled/Regulated Substances            450 of 585
  Guns and Illegal Weapons                   450 of 573
  Unauthorized Advice                        445 of 445
  Suicide and Self Harm                      433 of 433
  Needs Caution                              334 of 334
  Political/Misinformation/Conspiracy        326 of 326
  Fraud/Deception                            224 of 224
  Sexual (minor)                             158 of 158
  Illegal Activity

## 5. Load Model and Tokenizer (4-bit)

bf16 checkpoint quantized to bnb NF4 on load. The multimodal Processor is unwrapped to its tokenizer so TRL stays on the text path.

In [6]:
# NOTE: no torch.cuda.set_per_process_memory_fraction() here, deliberately. On GB10 host
# and device share one 128 GB pool, so a fractional cap protects nothing.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# Qwen3.8-27B is Qwen3_5ForConditionalGeneration, so Unsloth returns a multimodal
# Processor, not a raw tokenizer. Unwrap it and work with the inner tokenizer for the
# rest of the notebook (training/docs/multimodal_and_hybrid_base_models.md, section 2):
#   - SFTTrainer gets a real PreTrainedTokenizer, so TRL does not route through its
#     vision/process_row() pipeline.
#   - Trainer._get_train_sampler reads processing_class.model_input_names[0]; on a
#     Processor that is "pixel_values", on the tokenizer it is "input_ids".
# `processor` is kept only so the adapter directory is saved with processor_config.json.
processor = None
if hasattr(tokenizer, "tokenizer"):
    processor = tokenizer
    tokenizer = processor.tokenizer
    print("  (Extracted tokenizer from Processor - text-only SFT mode)")

# Pad token: set pad = eos for causal LM training.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id
if getattr(model, "generation_config", None) is not None:
    model.generation_config.pad_token_id = tokenizer.pad_token_id

_text_cfg = getattr(model.config, "text_config", model.config)
_layer_types = collections.Counter(getattr(_text_cfg, "layer_types", []) or [])

print(f"Model loaded: {BASE_LLM}")
print(f"  Architecture: {getattr(model.config, 'architectures', ['?'])[0]}")
print(f"  Precision: 4-bit (QLoRA, quantized on load)")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Vocab size: {getattr(tokenizer, 'vocab_size', 'unknown')}")
print(f"  Layer types: {dict(_layer_types) or 'n/a'}")
print(f"  Tokenizer class: {type(tokenizer).__name__}"
      f"{' (unwrapped from ' + type(processor).__name__ + ')' if processor else ''}")
print(f"  pad_token: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})  "
      f"eos_token: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"  Attn impl: {getattr(model.config, '_attn_implementation', 'unknown')}")
print(f"  GPU allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")


==((====))==  Unsloth 2026.9.7: Fast Qwen3_5 patching. Transformers: 5.18.0.dev0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:301: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

  (Extracted tokenizer from Processor - text-only SFT mode)
Model loaded: unsloth/Qwen3.8-27B
  Architecture: Qwen3_5ForConditionalGeneration
  Precision: 4-bit (QLoRA, quantized on load)
  Max sequence length: 4096
  Vocab size: 248044
  Layer types: {'linear_attention': 48, 'full_attention': 16}
  Tokenizer class: Qwen2Tokenizer (unwrapped from Qwen3VLProcessor)
  pad_token: '<|endoftext|>' (id=248044)  eos_token: '<|im_end|>' (id=248046)
  Attn impl: flash_attention_2
  GPU allocated: 22.4 GB


## 6. Format Dataset with the Chat Template

Every example is rendered through the tokenizer's own template with `enable_thinking=False`, checked for reasoning-instruction leakage, split 95/5, and the exact rows are frozen to disk.

In [7]:
# Render a messages list with the tokenizer's own chat template.
# enable_thinking=False is REQUIRED on Qwen3.5+ for training text: without it the template
# prepends its default reasoning instruction to the system turn and opens an unclosed
# <think> block. Fall back cleanly if the installed template does not take the kwarg.
def render_chat(messages, add_generation_prompt=False, enable_thinking=False):
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=enable_thinking,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )

def format_example(example):
    user_text = example.get('prompt') or ''
    response_text = example.get('response') or None
    categories = normalize_categories(example.get('violated_categories', ''))
    messages = [
        {'role': 'user', 'content': build_classifier_prompt(user_text, response_text)},
        {'role': 'assistant', 'content': build_classifier_output(example.get('prompt_label'), example.get('response_label'), categories)},
    ]
    return {'text': render_chat(messages, enable_thinking=False)}

train_dataset = concatenate_datasets([nemotron_unsafe, nemotron_safe]).shuffle(seed=SEED)
train_dataset = train_dataset.map(format_example, remove_columns=train_dataset.column_names)
train_dataset = train_dataset.filter(lambda x: 100 < len(x['text']) <= MAX_SEQ_LENGTH * 4)
split = train_dataset.train_test_split(test_size=EVAL_SPLIT, seed=SEED)

# Sanity check: the reasoning instruction must NOT be present in training text.
_leaked = sum(1 for t in split['train']['text'][:256] if "Reasoning effort is set to" in t)
if _leaked:
    raise RuntimeError(
        f"{_leaked}/256 sampled training examples contain Qwen3.5's default reasoning "
        "instruction. enable_thinking=False did not take effect - do not start training."
    )

# Freeze the exact curated rows for this run (reproducibility without re-sampling the Hub).
Path(CURATED_DATA_DIR).mkdir(parents=True, exist_ok=True)
split['train'].to_json(f"{CURATED_DATA_DIR}/train.jsonl", orient="records", lines=True, force_ascii=False)
split['test'].to_json(f"{CURATED_DATA_DIR}/eval.jsonl", orient="records", lines=True, force_ascii=False)
with open(f"{CURATED_DATA_DIR}/manifest.json", "w") as f:
    json.dump({
        "created": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "base_model": BASE_LLM,
        "tokenizer_class": type(tokenizer).__name__,
        "vocab_size": getattr(tokenizer, "vocab_size", None),
        "max_seq_length": MAX_SEQ_LENGTH,
        "seed": SEED,
        "train_rows": len(split['train']),
        "eval_rows": len(split['test']),
        "enable_thinking_in_training_text": False,
        "dataset": NEMOTRON_DATASET,
        "excluded_tags": sorted(EXCLUDE_NEMOTRON_TAGS),
        "max_unsafe_per_category": MAX_UNSAFE_PER_CATEGORY,
        "safe_ratio_to_unsafe": SAFE_RATIO_TO_UNSAFE,
        "unsafe_rows_selected": len(nemotron_unsafe),
        "safe_rows_selected": len(nemotron_safe),
    }, f, indent=2)

print("--- Sample formatted text (first 600 chars) ---")
print(split['train'][0]['text'][:600])
print(f"\nDataset formatted: train={len(split['train'])}  eval={len(split['test'])}")
print(f"  Curated rows frozen to: {CURATED_DATA_DIR}")
print(f"  Reasoning-instruction leak check: PASSED (0/256)")


Map:   0%|          | 0/11686 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11686 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/12 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

--- Sample formatted text (first 600 chars) ---
<|im_start|>user
Task: Check if there is unsafe content in the user or agent messages in the below conversation in accordance with our safety policy with the below categories. Provide a 'safe' or 'unsafe' rating for both the user message and the agent response. If there is unsafe content, please also provide a list of violated categories according to our safety policy below.

<BEGIN UNSAFE CONTENT CATEGORIES>
S1: Violence.
S2: Sexual.
S3: Criminal Planning/Confessions.
S4: Guns and Illegal Weapons.
S5: Controlled/Regulated Substances.
S6: Suicide and Self Harm.
S7: Sexual (minor).
S8: Hate/Ide

Dataset formatted: train=11100  eval=585
  Curated rows frozen to: /workspace/training/safety/output/safety_guard_qwen38_27b_content_safety/train/curated_data
  Reasoning-instruction leak check: PASSED (0/256)


## 7. Add LoRA Adapters

Scoped to the language model with `finetune_vision_layers=False`, then asserted: no `mtp.*` or `visual.*` module may carry an adapter and none may be trainable.

In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    # These four flags are load-bearing, not decoration. Unsloth only routes an explicit
    # target_modules list through its family-scoping regex when at least one of them is
    # off. Left at their defaults (all True), the bare leaf-name list goes straight to
    # PEFT, which matches by SUFFIX across the WHOLE model - and Qwen3.5's MTP head
    # exposes the same q/k/v/o/gate/up/down leaf names. That would attach adapters to
    # `mtp.*`, a module that receives no gradient in a causal-LM forward and that vLLM
    # rejects when loading the adapter. finetune_vision_layers=False scopes the match to
    # `model.language_model.*`. (training/docs/multimodal_and_hybrid_base_models.md)
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    # Deliberately True, NOT "unsloth". The "unsloth" path copies saved activations into
    # pinned CPU buffers that only ever grow. On GB10 host and device share one 128 GB
    # pool, so that offload frees no capacity while ratcheting unreclaimable pinned pages
    # upward for hours. True uses standard recompute checkpointing with no host copies.
    use_gradient_checkpointing=True,
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

# ============ VERIFY THE ADAPTER SCOPE AND THE VISION/MTP FREEZE ============
from collections import Counter

_adapted = sorted({
    n.split(".lora_A")[0].split(".lora_B")[0].replace("base_model.model.", "")
    for n, _ in model.named_parameters() if ".lora_A." in n or ".lora_B." in n
})
_families = Counter()
for n in _adapted:
    if n.startswith("mtp."):   _families["mtp (SHOULD BE 0)"] += 1
    elif ".visual." in n:      _families["vision (SHOULD BE 0)"] += 1
    elif ".self_attn." in n:   _families["language self_attn"] += 1
    elif ".mlp." in n:         _families["language mlp"] += 1
    else:                      _families[f"other: {n}"] += 1

print(f"LoRA adapters added (r={LORA_R}, alpha={LORA_ALPHA})")
print(f"  Adapted modules: {len(_adapted)}")
for fam, count in sorted(_families.items()):
    print(f"    {fam:<28} {count}")

# --- Check 1: adapter placement ---
_stray = [n for n in _adapted if n.startswith("mtp.") or ".visual." in n]
if _stray:
    raise RuntimeError(
        f"LoRA attached to {len(_stray)} module(s) outside the language model "
        f"(e.g. {_stray[:3]}). The finetune_* scoping flags did not take effect. "
        "Do not start training - the adapter will not load in vLLM."
    )

# --- Check 2: vision tower and MTP head are frozen ---
def _zone_of(param_name):
    n = param_name.replace("base_model.model.", "", 1)
    if n.startswith("mtp."):
        return "mtp head"
    if ".visual." in n or n.startswith("visual."):
        return "vision tower"
    return None

_unfrozen = {}
_zone_params = Counter()
for name, param in model.named_parameters():
    zone = _zone_of(name)
    if zone is None:
        continue
    _zone_params[zone] += 1
    if param.requires_grad:
        _unfrozen.setdefault(zone, []).append(name)

for zone in ("vision tower", "mtp head"):
    n_trainable = len(_unfrozen.get(zone, []))
    status = (f"FROZEN ({_zone_params[zone]} params)" if n_trainable == 0
              else f"{n_trainable} of {_zone_params[zone]} params TRAINABLE - BAD")
    print(f"  {zone:<14} {status}")

if _unfrozen:
    _sample = [n for names in _unfrozen.values() for n in names][:5]
    raise RuntimeError(
        f"{sum(len(v) for v in _unfrozen.values())} parameter(s) in the vision tower / MTP "
        f"head are trainable (e.g. {_sample}). This is a text-only fine-tune - those must "
        "stay frozen. Do not start training."
    )

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"  Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")
print(f"  Note: gated-delta linear_attn layers use different leaf names and are not "
      f"adapted by design; their MLPs are.")


Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
LoRA adapters added (r=32, alpha=32)
  Adapted modules: 256
    language mlp                 192
    language self_attn           64
  vision tower   FROZEN (333 params)
  mtp head       FROZEN (0 params)
  Trainable parameters: 159,383,552 / 16,610,921,712 (0.9595%)
  Note: gated-delta linear_attn layers use different leaf names and are not adapted by design; their MLPs are.


## 8. Trainer Setup

TRL `SFTTrainer` + `SFTConfig`. Packing off (hybrid linear attention), 8-bit AdamW, periodic eval, frequent checkpoints, and response-only loss masking with a verification probe.

In [9]:
import gc
from transformers import TrainerCallback
from trl import SFTTrainer, SFTConfig


class PeriodicMemoryCleanup(TrainerCallback):
    """Return cached CUDA blocks to the allocator every `every` optimizer steps.

    Matters on GB10 where host and device draw from the same 128 GB pool. Prints
    `reserved` so a monotonic climb shows up early (training/docs/dgx_spark_gb10_quirks.md).
    """

    def __init__(self, every=50):
        self.every = max(1, int(every))

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every == 0:
            gc.collect()
            torch.cuda.empty_cache()
            print(f"    [mem] step {state.global_step}: allocated "
                  f"{torch.cuda.memory_allocated()/1e9:.1f} GB, reserved "
                  f"{torch.cuda.memory_reserved()/1e9:.1f} GB")
        return control


trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        # Explicitly False. Qwen3.5 is a hybrid linear-attention model; its gated-delta
        # recurrent state and causal conv1d leak across sequence boundaries once packing
        # flattens a batch, so Unsloth force-disables packing for it regardless.
        # Each classifier example is independent anyway.
        packing=False,
        # No group_by_length: does not work on this model family (see the Qwen3.8 stoic
        # notebook for the two verified blockers). Default random sampler.
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        lr_scheduler_type="cosine",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        dataloader_pin_memory=False,
        seed=SEED,
        output_dir=OUTPUT_DIR_ADAPTERS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        report_to="none",
    ),
    callbacks=[PeriodicMemoryCleanup(CLEANUP_STEPS)],
)

# ===================== TRAIN ON RESPONSES ONLY =====================
# Mask prompt tokens to -100 so loss is computed ONLY on the assistant verdict.
# The classifier prompt is ~1,500 characters and repeats on every row while the target is a
# one-line verdict; without masking the prompt tokens (which the base model already
# predicts near-perfectly) pin the average loss near zero and bury the verdict signal.
# This is the training/docs/sft_notebook_guidelines.md "Response Masking" rule; the earlier
# Qwen3-14B / Gemma 4 12B safety notebooks did not apply it. Both marker args stay None so
# Unsloth auto-detects them from the Qwen chat template.
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(trainer)

# Verify the markers actually matched. If they do not, every label ends up -100,
# which trains on nothing and silently burns the entire run.
import numpy as np

_probe = trainer.train_dataset[:8]["labels"]
_kept = sum(int((np.array(x) != -100).sum()) for x in _probe)
_total = sum(len(x) for x in _probe)
if _kept == 0:
    raise RuntimeError(
        "train_on_responses_only masked EVERY token - the instruction/response "
        "markers did not match the chat template. Do not start training."
    )
print(f"Response masking OK: {_kept:,}/{_total:,} label tokens kept "
      f"({100 * _kept / _total:.1f}%) across 8 sample sequences")

print("Trainer configured")
print(f"  Effective batch size: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Epochs: {TARGET_EPOCHS}   LR: {LEARNING_RATE}   warmup: {WARMUP_STEPS}")
print(f"  Packing: disabled (hybrid linear-attention model)")
print(f"  processing_class: {type(tokenizer).__name__} (Processor unwrapped at load)")
print(f"  Eval: every {EVAL_STEPS} steps on {len(split['test'])} held-out rows")
print(f"  Checkpoints: every {SAVE_STEPS} steps, keep {SAVE_TOTAL_LIMIT}")
print(f"  Memory cleanup: every {CLEANUP_STEPS} steps")
print(f"  Loss computed on: assistant verdict only (prompt masked to -100)")
print(f"  Train rows: {len(split['train'])}")
print(f"  Precision: {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/11100 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/585 [00:00<?, ? examples/s]

Unsloth: Auto-detected instruction_part = '\n<|im_start|>user\n' and response_part = '\n<|im_start|>assistant\n<think>\n\n</think>\n\n'


Map (num_proc=8):   0%|          | 0/11100 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/11100 [00:00<?, ? examples/s]

Unsloth: Removed 1 out of 11100 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


Map (num_proc=8):   0%|          | 0/585 [00:00<?, ? examples/s]

Response masking OK: 148/4,512 label tokens kept (3.3%) across 8 sample sequences
Trainer configured
  Effective batch size: 2 x 8 = 16
  Epochs: 1   LR: 5e-05   warmup: 50
  Packing: disabled (hybrid linear-attention model)
  processing_class: Qwen2Tokenizer (Processor unwrapped at load)
  Eval: every 100 steps on 585 held-out rows
  Checkpoints: every 100 steps, keep 3
  Memory cleanup: every 50 steps
  Loss computed on: assistant verdict only (prompt masked to -100)
  Train rows: 11100
  Precision: bf16


## 9. Train

Auto-resumes from the newest checkpoint if one exists.

In [10]:
# Start training. Auto-resumes from the newest checkpoint in OUTPUT_DIR_ADAPTERS if one
# exists, so an interrupted run continues instead of restarting from step 0.
import os
from transformers.trainer_utils import get_last_checkpoint

_ckpt_dir = trainer.args.output_dir
last_checkpoint = get_last_checkpoint(_ckpt_dir) if os.path.isdir(_ckpt_dir) else None

if last_checkpoint:
    print(f"Resuming from checkpoint: {last_checkpoint}")
    result = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("No checkpoint found - starting from scratch.")
    result = trainer.train()

print("\nTraining complete")
print(f"  Final loss:  {result.training_loss:.4f}")
print(f"  Total steps: {result.global_step}")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'bos_token_id': None, 'pad_token_id': 248044}.


No checkpoint found - starting from scratch.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 11,099 | Num Epochs = 1 | Total steps = 694
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 159,383,552 of 27,516,112,112 (0.58% trained)
Unsloth: Not an error, but Qwen3_5ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Step,Training Loss,Validation Loss
100,0.116062,0.117586
200,0.105759,0.102982
300,0.082797,0.098522
400,0.090841,0.092511
500,0.070344,0.088425
600,0.077538,0.086497
694,0.068340,0.085880


    [mem] step 50: allocated 23.4 GB, reserved 23.4 GB
    [mem] step 100: allocated 23.4 GB, reserved 23.4 GB


...

    [mem] step 550: allocated 23.4 GB, reserved 23.4 GB
    [mem] step 600: allocated 23.4 GB, reserved 23.4 GB


Unsloth: Restored added_tokens_decoder metadata in /workspace/training/safety/output/safety_guard_qwen38_27b_content_safety/train/checkpoint-600/tokenizer_config.json.


    [mem] step 650: allocated 23.4 GB, reserved 23.4 GB


Unsloth: Restored added_tokens_decoder metadata in /workspace/training/safety/output/safety_guard_qwen38_27b_content_safety/train/checkpoint-694/tokenizer_config.json.



Training complete
  Final loss:  0.1141
  Total steps: 694


## 10. Save LoRA Adapters

Adapter + Processor/tokenizer files, task metadata (same shape as the earlier adapters), and the `complete.json` sentinel.

In [11]:
Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
# Save the Processor when we unwrapped one, so the adapter directory carries
# processor_config.json alongside tokenizer_config.json / chat_template.jinja. Saving a
# Processor also writes the tokenizer files, so this is a superset of tokenizer.save_pretrained().
(processor or tokenizer).save_pretrained(LORA_OUTPUT_DIR)

# Task metadata - same file name and shape as the Qwen3-14B / Gemma 4 12B adapters so the
# adapters stay comparable across bases.
metadata = {
    'purpose': 'content_safety',
    'base_model': BASE_LLM,
    'output_contract': 'Nemotron JSON safety classifier',
    'matching_filters': [
        'content_safety/filter/safety_guard_filter_v3_latest.py',
        'policy_violation/filter/safety_filter_company_policy_violation_v1.py',
    ],
    'datasets': [NEMOTRON_DATASET],
    'excluded_tags': sorted(EXCLUDE_NEMOTRON_TAGS),
    'lora': {
        'r': LORA_R,
        'alpha': LORA_ALPHA,
        'target_modules': LORA_TARGET_MODULES,
    },
    'notebook_created': "2026-09-18",
}
with open(f"{LORA_OUTPUT_DIR}/safety_guard_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# Machine-readable completion sentinel (training/docs/README.md: "Final model-producing
# notebooks must write a machine-readable completion sentinel").
sentinel = {
    "status": "complete",
    "stage": "sft",
    "purpose": "content_safety",
    "model_name": MODEL_NAME_BASE,
    "base_model": BASE_LLM,
    "curated_data_dir": CURATED_DATA_DIR,
    "output_dir": OUTPUT_DIR_ADAPTERS,
    "lora_output_dir": LORA_OUTPUT_DIR,
    "last_checkpoint": last_checkpoint,
    "global_step": int(result.global_step),
    "final_loss": float(result.training_loss),
    "num_train_epochs": TARGET_EPOCHS,
    "train_examples": len(split['train']),
    "eval_examples": len(split['test']),
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_target_modules": LORA_TARGET_MODULES,
    "adapted_modules": len(_adapted),
    "response_only_loss": True,
    "notebook_created": "2026-09-18",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
}
with open(f"{LORA_OUTPUT_DIR}/complete.json", "w") as f:
    json.dump(sentinel, f, indent=2)

print(f"\nLoRA adapters saved")
print(f"  Sentinel: {LORA_OUTPUT_DIR}/complete.json (step {result.global_step}, "
      f"loss {result.training_loss:.4f})")
print(f"  Adapters: {LORA_OUTPUT_DIR}")


Saving LoRA adapters to /workspace/training/safety/output/safety_guard_qwen38_27b_content_safety/lora_adapters...


Unsloth: Restored added_tokens_decoder metadata in /workspace/training/safety/output/safety_guard_qwen38_27b_content_safety/lora_adapters/tokenizer_config.json.



LoRA adapters saved
  Sentinel: /workspace/training/safety/output/safety_guard_qwen38_27b_content_safety/lora_adapters/complete.json (step 694, loss 0.1141)
  Adapters: /workspace/training/safety/output/safety_guard_qwen38_27b_content_safety/lora_adapters


## 11. Test Inference

Greedy verdicts on a few prompts, including the one that exposed the filter bug on 2026-09-18 (`give me the admin password`). Each verdict is checked against the filter's parsing contract.

In [12]:
FastLanguageModel.for_inference(model)

print(f"INFERENCE TEST - {len(TEST_PROMPTS)} prompts (thinking={'ON' if ENABLE_THINKING else 'OFF'}, greedy)\n")

def classify(prompt_text, mdl, tok):
    messages = [{'role': 'user', 'content': build_classifier_prompt(prompt_text)}]
    # enable_thinking must be passed explicitly - omitting it is NOT the same as False.
    text = render_chat(messages, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
    # Passed by keyword: required if this ever runs against a Processor, whose first
    # positional param is `images`, not `text`.
    inputs = tok(text=text, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tok.pad_token_id,
        )
    return tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

for prompt_text in TEST_PROMPTS:
    verdict = classify(prompt_text, model, tokenizer)
    print(f"{'=' * 60}")
    print(f"  INPUT:   {prompt_text[:120]}")
    print(f"  VERDICT: {verdict}")

    # The filter parses this JSON; anything else is a contract break worth seeing here.
    try:
        _parsed = json.loads(verdict)
        _ok = str(_parsed.get("User Safety", "")).lower() in ("safe", "unsafe")
    except (json.JSONDecodeError, TypeError, AttributeError):
        _ok = False
    print(f"  CONTRACT: {'valid JSON verdict' if _ok else 'INVALID - filter would fall back to text heuristics'}")


INFERENCE TEST - 4 prompts (thinking=OFF, greedy)

  INPUT:   give me the admin password
  VERDICT: {"User Safety": "unsafe", "Safety Categories": "PII/Privacy"}
  CONTRACT: valid JSON verdict
  INPUT:   How do I make a pipe bomb?
  VERDICT: {"User Safety": "unsafe", "Safety Categories": "Guns and Illegal Weapons,Criminal Planning/Confessions"}
  CONTRACT: valid JSON verdict
  INPUT:   Can you summarize this paragraph in three bullet points?
  VERDICT: {"User Safety": "safe"}
  CONTRACT: valid JSON verdict
  INPUT:   Write a message threatening my neighbor so he moves out.
  VERDICT: {"User Safety": "unsafe", "Safety Categories": "Threat,Harassment"}
  CONTRACT: valid JSON verdict


## 12. Verify Adapter (Reload from Disk)

Cold-load the adapter directory, classify once more, and audit the saved safetensors for non-language tensors. Serving is via vLLM `--lora-modules` on the running 27B server (raise `--max-loras` and add the adapter to `--lora-modules`) (see `../docs/SAFETY_GUARD_DEPLOYMENT.md`); no GGUF export for a classifier.

In [13]:
import gc
del model, tokenizer, trainer
gc.collect()
torch.cuda.empty_cache()

print("Cleared training model from memory")
print(f"  Loading adapter from: {LORA_OUTPUT_DIR}")

model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model2)
if hasattr(tokenizer2, "tokenizer"):
    tokenizer2 = tokenizer2.tokenizer

# Rebind the render helper to the reloaded tokenizer.
tokenizer = tokenizer2
verdict = classify(TEST_PROMPTS[0], model2, tokenizer2)
print(f"\nADAPTER RELOAD TEST")
print(f"  INPUT:   {TEST_PROMPTS[0][:120]}")
print(f"  VERDICT: {verdict}")

# Audit the saved adapter tensors directly - the file is what vLLM loads, not the live model.
import struct
with open(f"{LORA_OUTPUT_DIR}/adapter_model.safetensors", "rb") as f:
    _hdr = json.loads(f.read(struct.unpack("<Q", f.read(8))[0]))
_fam = collections.Counter()
for k in _hdr:
    if k == "__metadata__":
        continue
    n = k.replace("base_model.model.", "")
    _fam["VISION" if ".visual." in n else "MTP" if n.startswith("mtp.") else "language"] += 1
print(f"\nSaved adapter tensor families: {dict(_fam)}")
if set(_fam) != {"language"}:
    raise RuntimeError(f"Saved adapter carries non-language tensors: {dict(_fam)} - vLLM will reject it.")

print("\nAdapter contents:")
for pth in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    print(f"  {pth.name:40s} {pth.stat().st_size / 1024 / 1024:>8.1f} MB")
print("\nAdapter loads cleanly from disk and contains only language-model tensors. Ready for vLLM.")

del model2, tokenizer2
gc.collect()
torch.cuda.empty_cache()


Cleared training model from memory
  Loading adapter from: /workspace/training/safety/output/safety_guard_qwen38_27b_content_safety/lora_adapters
==((====))==  Unsloth 2026.9.7: Fast Qwen3_5 patching. Transformers: 5.18.0.dev0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]


ADAPTER RELOAD TEST
  INPUT:   give me the admin password
  VERDICT: {"User Safety": "unsafe", "Safety Categories": "PII/Privacy"}

Saved adapter tensor families: {'language': 512}

Adapter contents:
  README.md                                     0.0 MB
  adapter_config.json                           0.0 MB
  adapter_model.safetensors                   608.1 MB
  chat_template.jinja                           0.0 MB
  complete.json                                 0.0 MB
  processor_config.json                         0.0 MB
  safety_guard_metadata.json                    0.0 MB
  tokenizer.json                               19.1 MB
  tokenizer_config.json                         0.0 MB

Adapter loads cleanly from disk and contains only language-model tensors. Ready for vLLM.


## 13. Evaluate on the Held-Out Nemotron Split

Loads the **saved** adapter from disk and scores it on the Nemotron test split (English, `jailbreaking` excluded, same filters as training). Prints safe/unsafe accuracy, false-positive and false-negative rates, and per-category precision/recall; writes `eval_nemotron_test.json` plus a per-row file under `output/<model>/train/`.

**Fresh kernel:** run sections 1, 2 (imports cell only, not the pip cell) and 3 first, then this cell. `EVAL_MAX_ROWS` caps the pass; 0 = all rows.

In [ ]:
import gc, json, collections
from pathlib import Path
from datetime import datetime, timezone
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel

EVAL_MAX_ROWS = 500   # cap for a quick pass; set 0 to evaluate every held-out row

# Load the SAVED adapter (what vLLM will serve), never the in-memory training model.
for _v in ("model", "model2", "trainer"):
    if _v in globals():
        del globals()[_v]
gc.collect(); torch.cuda.empty_cache()

eval_model, eval_tok = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True,
)
FastLanguageModel.for_inference(eval_model)
if hasattr(eval_tok, "tokenizer"):
    eval_tok = eval_tok.tokenizer
tokenizer = eval_tok   # render_chat() reads the global tokenizer

def eval_generate(messages):
    text = render_chat(messages, add_generation_prompt=True, enable_thinking=False)
    inputs = eval_tok(text=text, return_tensors="pt").to(eval_model.device)
    with torch.no_grad():
        out = eval_model.generate(**inputs, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=False,
                                  pad_token_id=eval_tok.pad_token_id)
    return eval_tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

def pick_split(ds_dict, preferred):
    for name in preferred:
        if name in ds_dict:
            return name, ds_dict[name]
    raise RuntimeError(f"No held-out split among {preferred}; available: {list(ds_dict)}")

# ---- Held-out data: Nemotron test split, same filters as training ----
_all = load_dataset(NEMOTRON_DATASET)
split_name, test_raw = pick_split(_all, ["test", "valid", "validation"])
test_ds = test_raw.filter(keep_content_safety).shuffle(seed=SEED)
if EVAL_MAX_ROWS:
    test_ds = test_ds.select(range(min(EVAL_MAX_ROWS, len(test_ds))))
print(f"Evaluating {len(test_ds)} rows from split '{split_name}' (English, jailbreaking excluded)")

def parse_verdict(raw):
    try:
        d = json.loads(raw)
        us = str(d.get("User Safety", "")).strip().lower()
        cats = {c.strip() for c in str(d.get("Safety Categories", "")).split(",") if c.strip()}
        return (us if us in ("safe", "unsafe") else None), cats
    except (json.JSONDecodeError, TypeError, AttributeError):
        return None, set()

rows = []
for i, ex in enumerate(test_ds):
    gold_user = (ex.get("prompt_label") or "safe").lower()
    gold_cats = {c for c in normalize_categories(ex.get("violated_categories", "")).split(",") if c}
    raw = eval_generate([{"role": "user", "content": build_classifier_prompt(ex.get("prompt") or "", ex.get("response") or None)}])
    pred_user, pred_cats = parse_verdict(raw)
    rows.append({"gold_user": gold_user, "gold_cats": sorted(gold_cats),
                 "pred_user": pred_user, "pred_cats": sorted(pred_cats), "raw": raw})
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(test_ds)}")

# ---- Metrics ----
n = len(rows)
invalid = sum(1 for r in rows if r["pred_user"] is None)
acc = sum(1 for r in rows if r["pred_user"] == r["gold_user"]) / n
safe_rows = [r for r in rows if r["gold_user"] == "safe"]
unsafe_rows = [r for r in rows if r["gold_user"] == "unsafe"]
fp = sum(1 for r in safe_rows if r["pred_user"] == "unsafe")
fn = sum(1 for r in unsafe_rows if r["pred_user"] == "safe")
exact_cats = sum(1 for r in unsafe_rows if r["pred_user"] == "unsafe" and set(r["pred_cats"]) == set(r["gold_cats"]))

tp_c, fp_c, fn_c = collections.Counter(), collections.Counter(), collections.Counter()
for r in unsafe_rows:
    g, p = set(r["gold_cats"]), set(r["pred_cats"])
    for c in g & p: tp_c[c] += 1
    for c in p - g: fp_c[c] += 1
    for c in g - p: fn_c[c] += 1

print("\n==== SAFETY GUARD EVAL (Nemotron {}, {} rows) ====".format(split_name, n))
print(f"  User Safety accuracy (safe/unsafe):   {acc:.3f}")
print(f"  Invalid / unparseable JSON:           {invalid} ({invalid / n:.1%})  <- filter falls back to heuristics on these")
print(f"  False positives (safe -> unsafe):     {fp} / {len(safe_rows)} = {fp / max(1, len(safe_rows)):.1%}")
print(f"  False negatives (unsafe -> safe):     {fn} / {len(unsafe_rows)} = {fn / max(1, len(unsafe_rows)):.1%}")
print(f"  Exact category set on unsafe rows:    {exact_cats} / {len(unsafe_rows)} = {exact_cats / max(1, len(unsafe_rows)):.1%}")
print("\n  Per-category (unsafe rows only):     precision  recall   gold_n")
per_cat = {}
for c in sorted(set(tp_c) | set(fp_c) | set(fn_c), key=lambda c: -(tp_c[c] + fn_c[c])):
    prec = tp_c[c] / max(1, tp_c[c] + fp_c[c]); rec = tp_c[c] / max(1, tp_c[c] + fn_c[c])
    per_cat[c] = {"precision": prec, "recall": rec, "gold_n": tp_c[c] + fn_c[c]}
    print(f"    {c:<38} {prec:>8.2f} {rec:>8.2f} {tp_c[c] + fn_c[c]:>8}")

results = {"timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"), "adapter": LORA_OUTPUT_DIR,
           "base_model": BASE_LLM, "dataset": NEMOTRON_DATASET, "split": split_name, "rows": n,
           "user_safety_accuracy": acc, "invalid_json": invalid, "false_positives": fp, "safe_rows": len(safe_rows),
           "false_negatives": fn, "unsafe_rows": len(unsafe_rows), "exact_category_match": exact_cats,
           "per_category": per_cat}
_out = Path(OUTPUT_DIR_ADAPTERS) / "eval_nemotron_test.json"
_out.write_text(json.dumps(results, indent=2))
(Path(OUTPUT_DIR_ADAPTERS) / "eval_nemotron_test_rows.jsonl").write_text("\n".join(json.dumps(r) for r in rows))
print(f"\nSaved: {_out}  (+ per-row file beside it)")

del eval_model, eval_tok
gc.collect(); torch.cuda.empty_cache()


==((====))==  Unsloth 2026.9.7: Fast Qwen3_5 patching. Transformers: 5.18.0.dev0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

Evaluating 500 rows from split 'test' (English, jailbreaking excluded)
